In [1]:
from actor_system import PipelinedActor, ActorSystem
from msg_type import SIB1Decoded, SIB1Fail, BufferRelease

In [2]:
class SIB1DecodeActor(PipelinedActor):
    """SIB1 decode. Pipeline: CRS → PCFICH → PDCCH → PDSCH → DLSCH.
    
    PDSCH and DLSCH only run if PDCCH finds DCI (dci_decoded=True).
    """

    def __init__(self, system, buf, pool, graph, node_pipeline):
        super().__init__('sib1_decode', system, pool, graph, node_pipeline[0])
        self._buf = buf
        self._pipeline = node_pipeline

    def _fill_slot(self, slot_idx, msg):
        slot = self._pool.slots[slot_idx]
        data = self._buf.read_protected(msg.pid)
        slot.data[:len(data)] = data
        slot.tag = msg.tag
        slot.N_id = msg.tag['N_id']
        slot.f_d = msg.tag['f_d']
        slot.ns = msg.tag['ns']
        slot.n_ant = msg.tag['n_ant']
        slot.phich_res = msg.tag['phich_res']
        slot.sfn = msg.tag['sfn']
        self.system.send_message('buffer_manager', BufferRelease(pid=msg.pid))

    def on_result(self, msg):
        slot = self._pool.slots[msg.slot]
        if slot.sib1_decoded:
            self.system.send_message('controller', SIB1Decoded(
                sib1_bytes=slot.sib1_bytes, tag=msg.tag))
        else:
            self.system.send_message('controller', SIB1Fail(tag=msg.tag))

    def on_config(self, msg):
        for node in self._pipeline:
            if node.name in msg.params:
                node.func.config(**msg.params[node.name])

#### Test
1. cell search actor 

In [3]:
from data.lte_system_info import LTEParams
from flow_graph import FlowGraph, FunctionNode, ResizableSlotPool, SlotPool
from cell_search import PSSDetection, SSSDetection
import time
from actor_system import Actor
from msg_type import PipelineDone
import numpy as np
from slot_type import CellSearchSlot
from buffer_manager_actor import BufferManagerActor
from cell_search_actor import CellSearchActor
from msg_type import BufferRead, CellFound, NoCell

In [4]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [5]:
class CollectorActor(Actor):
    def __init__(self, name, system):
        super().__init__(name, system)
        self.messages = []
    def _default_behavior(self, message):
        self.messages.append(message)

system = ActorSystem()
graph = FlowGraph(num_workers=4)

pool = ResizableSlotPool(6, lambda: CellSearchSlot(params.N_subframe))

pss_func = PSSDetection(params, peak_ratio=5.0)
sss_func = SSSDetection(params, peak_ratio=8.0)

pss_node = FunctionNode('pss', pool.make_stage(pss_func), concurrency=1)
sss_node = FunctionNode('sss', pool.make_stage(sss_func), concurrency=1,
                        done_callback=lambda token: system.send_message(
                            'cell_search', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(pss_node, sss_node)

ctrl = CollectorActor('controller', system)
bm = BufferManagerActor(system, rxf, buffer_size=len(rxf),
                        batch_size=len(rxf), ingest_delay=0.0)
cs = CellSearchActor(system, bm.buf, pool, graph, params, [pss_node, sss_node])

system.create_actor(ctrl)
system.create_actor(cs)
system.create_actor(bm)

system.send_message('buffer_manager', 'start')

# ---- initial search ----
stride = params.stride
pos = 0
chunk_id = 0
while pos + params.N_subframe <= params.N_half_frame:
    system.send_message('buffer_manager', BufferRead(
        offset=pos, length=params.N_subframe, dest='cell_search',
        tag={'mode': 'initial_search', 'chunk_tag': chunk_id, 'pos': pos}))
    pos += stride
    chunk_id += 1

time.sleep(0.5)

# ---- tracking: one half-frame later ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
msg = cell_found[0]
pss_global = msg.tag['pos'] + msg.pss_local_index

expected_pss = pss_global + params.N_half_frame
track_start = expected_pss - 2 * params.N_ofdm_sym
track_len = params.pss_tracking_len
track_offset = track_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=track_offset, length=track_len, dest='cell_search',
    tag={'mode': 'tracking', 'frame_tag': 0}))

time.sleep(0.5)

# ---- print results ----
print(f'\nController received {len(ctrl.messages)} messages:')
for m in ctrl.messages:
    if isinstance(m, CellFound):
        if m.tag.get('mode') == 'initial_search':
            pss_g = m.tag['pos'] + m.pss_local_index
            print(f'  CellFound (initial): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
        elif m.tag.get('mode') == 'tracking':
            pss_g = track_start + m.pss_local_index
            print(f'  CellFound (tracking): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
    elif isinstance(m, NoCell):
        print(f'  NoCell: {m.tag.get("mode")}')
    else:
        print(f'  {m}')


Controller received 5 messages:
  start
  NoCell: initial_search
  NoCell: initial_search
  CellFound (initial): PCI=380, F=0, f_d=1126.9, pss_global=36043
  CellFound (tracking): PCI=380, F=1, f_d=1128.4, pss_global=112842


In [6]:
from mib_decode import PBCHDecoding, BCHDecoding
from crs_estimate import CRSChannelEstimation
from slot_type import MIBSlot
from mib_decode_actor import MIBDecodeActor
from msg_type import Config, MIBDecoded, MIBFail

In [7]:
# ---- extract cell search results ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
msg = cell_found[0]
N_id = msg.N_id
f_d = msg.f_d
pss_global = msg.tag['pos'] + msg.pss_local_index

print(f"Cell found: PCI={N_id}, f_d={f_d:.1f} Hz, pss_global={pss_global}")

# ---- MIB pipeline on SAME graph ----
mib_pool = SlotPool(4, lambda: MIBSlot(params.pbch_len))

crs_func = CRSChannelEstimation(params)
pbch_func = PBCHDecoding()
bch_func = BCHDecoding()

crs_node  = FunctionNode('mib_crs',  mib_pool.make_stage(crs_func),  concurrency=1)
pbch_node = FunctionNode('mib_pbch', mib_pool.make_stage(pbch_func), concurrency=1)
bch_node  = FunctionNode('mib_bch',  mib_pool.make_stage(bch_func),  concurrency=1,
                         done_callback=lambda token: system.send_message(
                             'mib_decode', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(crs_node, pbch_node)
graph.add_edge(pbch_node, bch_node)

mib = MIBDecodeActor(system, bm.buf, mib_pool, graph, [crs_node, pbch_node, bch_node])
system.create_actor(mib)

# ---- config ----
system.send_message('mib_decode', Config(params={
    'mib_crs':  {'N_id': N_id, 'N_rb': 6, 'ns': 1,
                 'n_symbols': 4, 'is_slot_start': True},
    'mib_pbch': {'N_id': N_id},
    'mib_bch':  {'N_id': N_id}
}))
time.sleep(0.1)

# ==== Step 1: first MIB decode (blind trial) ====

slot1_start = pss_global + params.N_FFT
slot1_offset = slot1_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=slot1_offset, length=params.pbch_len, dest='mib_decode',
    tag={'N_id': N_id, 'f_d': f_d, 'frame_tag': 0}))

time.sleep(1)

mib_msgs = [m for m in ctrl.messages if isinstance(m, MIBDecoded)]
mib_fails = [m for m in ctrl.messages if isinstance(m, MIBFail)]
print(f'\nMIBDecoded: {len(mib_msgs)}, MIBFail: {len(mib_fails)}')
assert len(mib_msgs) == 1, f"Expected 1 MIBDecoded, got {len(mib_msgs)}"

r = mib_msgs[0]
print(f"\n=== MIB #1 (blind trial) ===")
print(f"  n_ant={r.n_ant}, BW={r.dl_bw}, SFN={r.sfn}")
print(f"  PHICH dur={r.phich_dur}, res={r.phich_res}")
print(f"  cached n_ant = {mib._known_n_ant}")

# ==== Step 2: tracking PSS for next even-SFN frame ====

sfn = r.sfn
delta_frames = 2 if sfn % 2 == 0 else 1
next_even_pss = pss_global + delta_frames * params.N_frame

track_start = next_even_pss - 2 * params.N_ofdm_sym
track_len = params.pss_tracking_len
track_offset = track_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=track_offset, length=track_len, dest='cell_search',
    tag={'mode': 'tracking', 'frame_tag': 1}))

time.sleep(0.5)

tracking_found = [m for m in ctrl.messages if isinstance(m, CellFound)
                  and m.tag.get('mode') == 'tracking'
                  and m.tag.get('frame_tag') == 1]
assert len(tracking_found) == 1, f"Expected 1 tracking CellFound, got {len(tracking_found)}"

track_pss_global = track_start + tracking_found[0].pss_local_index
track_f_d = tracking_found[0].f_d

print(f"\n=== Tracking PSS ===")
print(f"  pss_pos={track_pss_global}, "
      f"drift={track_pss_global - next_even_pss}, f_d={track_f_d:.1f} Hz")

# ==== Step 3: second MIB decode (fast path) ====

slot1_start_2 = track_pss_global + params.N_FFT
mib_offset_2 = slot1_start_2 - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=mib_offset_2, length=params.pbch_len, dest='mib_decode',
    tag={'N_id': N_id, 'f_d': track_f_d, 'frame_tag': 1}))

time.sleep(0.5)

mib_msgs_2 = [m for m in ctrl.messages if isinstance(m, MIBDecoded)]
assert len(mib_msgs_2) == 2, f"Expected 2 MIBDecoded, got {len(mib_msgs_2)}"

r2 = mib_msgs_2[1]
print(f"\n=== MIB #2 (fast path) ===")
print(f"  n_ant={r2.n_ant}, BW={r2.dl_bw}, SFN={r2.sfn}")
print(f"  PHICH dur={r2.phich_dur}, res={r2.phich_res}")

Cell found: PCI=380, f_d=1126.9 Hz, pss_global=36043

MIBDecoded: 1, MIBFail: 0

=== MIB #1 (blind trial) ===
  n_ant=2, BW=50, SFN=313
  PHICH dur=normal, res=1
  cached n_ant = 2

=== Tracking PSS ===
  pss_pos=189643, drift=0, f_d=1164.2 Hz

=== MIB #2 (fast path) ===
  n_ant=2, BW=50, SFN=314
  PHICH dur=normal, res=1


In [8]:
from slot_type import SIB1Slot
from sib1_decode_control import PCFICHDecoding, PDCCHDecoding
from sib1_decode_data import PDSCHDecoding, DLSCHDecoding
from msg_type import SIB1Decoded, SIB1Fail, BufferRead, Config

In [9]:
# ---- extract MIB results from even-SFN frame ----
n_ant = r2.n_ant
dl_bw = r2.dl_bw
phich_res = r2.phich_res
sfn_even = r2.sfn

print(f"MIB (even): SFN={sfn_even}, BW={dl_bw}, n_ant={n_ant}, phich_res={phich_res}")

# ---- SIB1 pipeline on SAME graph ----
sib1_pool = SlotPool(4, lambda: SIB1Slot(params.N_subframe, n_symbols=14, N_rb=dl_bw))

sib1_crs_func    = CRSChannelEstimation(params)
sib1_pcfich_func = PCFICHDecoding()
sib1_pdcch_func  = PDCCHDecoding()
sib1_pdsch_func  = PDSCHDecoding()
sib1_dlsch_func  = DLSCHDecoding()

sib1_crs_node    = FunctionNode('sib1_crs',    sib1_pool.make_stage(sib1_crs_func),    concurrency=1)
sib1_pcfich_node = FunctionNode('sib1_pcfich', sib1_pool.make_stage(sib1_pcfich_func), concurrency=1)
sib1_pdcch_node  = FunctionNode('sib1_pdcch',  sib1_pool.make_stage(sib1_pdcch_func),  concurrency=1)
sib1_pdsch_node  = FunctionNode('sib1_pdsch',  sib1_pool.make_stage(sib1_pdsch_func),  concurrency=1)
sib1_dlsch_node  = FunctionNode('sib1_dlsch',  sib1_pool.make_stage(sib1_dlsch_func),  concurrency=1,
                                done_callback=lambda token: system.send_message(
                                    'sib1_decode', PipelineDone(slot=token.slot, tag=token.tag)))

graph.add_edge(sib1_crs_node, sib1_pcfich_node)
graph.add_edge(sib1_pcfich_node, sib1_pdcch_node)
graph.add_edge(sib1_pdcch_node, sib1_pdsch_node)
graph.add_edge(sib1_pdsch_node, sib1_dlsch_node)

sib1_pipeline = [sib1_crs_node, sib1_pcfich_node, sib1_pdcch_node,
                 sib1_pdsch_node, sib1_dlsch_node]

sib1 = SIB1DecodeActor(system, bm.buf, sib1_pool, graph, sib1_pipeline)
system.create_actor(sib1)

# ---- config SIB1 pipeline ----
system.send_message('sib1_decode', Config(params={
    'sib1_crs':    {'N_id': N_id, 'N_rb': dl_bw, 'ns': 10,
                    'n_symbols': 14, 'is_slot_start': True},
    'sib1_pcfich': {'N_id': N_id, 'N_rb': dl_bw, 'ns': 10},
    'sib1_pdcch':  {'N_id': N_id, 'N_rb': dl_bw,
                    'n_ant': n_ant, 'phich_res': phich_res, 'ns': 10},
    'sib1_pdsch':  {'N_id': N_id, 'n_ant': n_ant},
    'sib1_dlsch':  {'ns': 10, 'N_id': N_id}
}))
time.sleep(0.1)

# ---- locate subframe 5 of the even-SFN frame ----
p = params
slot1_start_even = track_pss_global + p.N_FFT
frame_start_even = slot1_start_even - p.N_slot
subf5_start = frame_start_even + 5 * p.N_subframe

sib1_offset = subf5_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=sib1_offset, length=p.N_subframe, dest='sib1_decode',
    tag={'N_id': N_id, 'f_d': track_f_d, 'n_ant': n_ant,
         'ns': 10, 'phich_res': phich_res, 'sfn': sfn_even}))

time.sleep(2)

# ---- check results ----
sib1_ok = [m for m in ctrl.messages if isinstance(m, SIB1Decoded)]
sib1_fail = [m for m in ctrl.messages if isinstance(m, SIB1Fail)]
print(f'\nSIB1Decoded: {len(sib1_ok)}, SIB1Fail: {len(sib1_fail)}')

assert len(sib1_ok) == 1, f"Expected 1 SIB1Decoded, got {len(sib1_ok)} (fails: {len(sib1_fail)})"

s = sib1_ok[0]
print(f"\n=== SIB1 Decoded ===")
print(f"  SFN:        {sfn_even}")
print(f"  SIB1 bytes: {s.sib1_bytes[:20]}...")
print(f"  Length:     {len(s.sib1_bytes)} bytes")

graph.shutdown()  # shutdown everything at the very end

MIB (even): SFN=314, BW=50, n_ant=2, phich_res=1

SIB1Decoded: 1, SIB1Fail: 0

=== SIB1 Decoded ===
  SFN:        314
  SIB1 bytes: [ 72  72  80   3   1  22  70   7  64 120  25  49  16 129   4  76  35 203
  82   0]...
  Length:     22 bytes
